# TAAF ARC-AGI-3 Kaggle Run

This notebook is generated by the Tufa ARC-AGI Framework (TAAF), an open-source deployment harness from [Tufa Labs](https://tufalabs.ai/) for running ARC-AGI-3 solvers reproducibly on Kaggle.

The notebook installs the ARC runtime, makes the bundled TAAF source snapshot importable, runs any solver setup commands, loads the pickled benchmark, and writes results to `/kaggle/working`. It can run as a public/offline debug notebook or as the same code path used for competition reruns. Kaggle's `KAGGLE_IS_COMPETITION_RERUN` flag always wins and switches the run into submission mode.

For quick inline experiments, use the customization code cell just before the benchmark run. That is the safest place to tweak the benchmark or solver after the deployed bundle has loaded.


In [1]:
import contextlib
import dataclasses
import json
import os
import pickle
import subprocess
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path
from typing import TextIO
from urllib.request import urlopen


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name, "").strip().lower()
    if not raw:
        return default
    return raw in {"1", "true", "yes", "y", "on"}


NOTEBOOK_START_EPOCH = time.time()
RUN_AS_SUBMISSION = False
RUN_AS_SUBMISSION = RUN_AS_SUBMISSION or _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
ENABLE_GPU = True

os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if RUN_AS_SUBMISSION else "0"
os.environ.setdefault("MPLBACKEND", "Agg")

if ENABLE_GPU:
    cuda_library_path = "/usr/local/nvidia/lib64"
    existing = [entry for entry in os.environ.get("LIBRARY_PATH", "").split(os.pathsep) if entry]
    os.environ["LIBRARY_PATH"] = os.pathsep.join(
        [cuda_library_path, *[entry for entry in existing if entry != cuda_library_path]]
    )

print(f"TAAF RUN_AS_SUBMISSION={RUN_AS_SUBMISSION}")
if ENABLE_GPU:
    print(f"taaf.kaggle: LIBRARY_PATH={os.environ['LIBRARY_PATH']}")

TAAF RUN_AS_SUBMISSION=False
taaf.kaggle: LIBRARY_PATH=/usr/local/nvidia/lib64:/usr/local/cuda/lib64/stubs


In [2]:
wheelhouse = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels")
if wheelhouse.exists():
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--no-index",
            "--no-warn-conflicts",
            "--disable-pip-version-check",
            "--find-links",
            str(wheelhouse),
            "arc-agi",
        ]
    )
elif os.getenv("TAAF_KAGGLE_BUNDLE_DIR"):
    print(f"Competition wheelhouse not found at {wheelhouse}; assuming local debug dependencies are installed.")
else:
    raise RuntimeError(f"Competition wheelhouse not found at {wheelhouse}.")

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0


In [3]:
DATASET_SOURCES: list[str] = ["lordhansolo/taaf-kaggle-source", "lordhansolo/vllm-main-e975732-arc3"]
KERNEL_SOURCES: list[str] = []
DATASET_BUNDLE_MARKER = "taaf-kaggle-bundle.json"
WORKING_DIR = Path(os.getenv("TAAF_KAGGLE_WORKING_DIR", "/kaggle/working")).resolve()
SETUP_ENV_PATH = WORKING_DIR / "taaf_setup_env.json"
SOFT_DEADLINE_BUFFER_S = 600.0
WORKING_DIR.mkdir(parents=True, exist_ok=True)


def _split_ref(ref: str) -> tuple[str, str]:
    owner, slug = ref.split("/", 1)
    return owner, slug


def _dataset_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/input") / slug, Path("/kaggle/input/datasets") / owner / slug]


def _kernel_mount_candidates(ref: str) -> list[Path]:
    owner, slug = _split_ref(ref)
    return [Path("/kaggle/usr/lib/notebooks") / owner / slug]


def _first_existing(candidates: list[Path]) -> Path | None:
    return next((candidate for candidate in candidates if candidate.exists()), None)


def _find_taaf_bundle() -> Path:
    explicit = os.getenv("TAAF_KAGGLE_BUNDLE_DIR", "").strip()
    if explicit:
        path = Path(explicit)
        if (path / DATASET_BUNDLE_MARKER).is_file():
            return path
    for root in [Path("/kaggle/input"), Path.cwd()]:
        if root.exists():
            for marker in root.rglob(DATASET_BUNDLE_MARKER):
                return marker.parent
    raise RuntimeError("Could not find TAAF Kaggle source bundle dataset.")


def _load_setup_env() -> dict[str, str]:
    if not SETUP_ENV_PATH.is_file():
        return {}
    data = json.loads(SETUP_ENV_PATH.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise RuntimeError(f"{SETUP_ENV_PATH} must contain a JSON object.")
    return {str(key): str(value) for key, value in data.items()}


def _write_setup_env_updates(updates: dict[str, str]) -> None:
    data = _load_setup_env()
    data.update(updates)
    SETUP_ENV_PATH.write_text(json.dumps(data, indent=2, sort_keys=True) + "\n", encoding="utf-8")


BUNDLE_DIR = _find_taaf_bundle()
print(f"TAAF source bundle: {BUNDLE_DIR}")

# Tell setup commands and solver code where Kaggle mounted every attached input.
kaggle_input_paths: dict[str, str] = {}
for index, ref in enumerate(DATASET_SOURCES):
    candidates = _dataset_mount_candidates(ref)
    resolved = BUNDLE_DIR if index == 0 else _first_existing(candidates)
    kaggle_input_paths[ref] = str(resolved or candidates[0])
for ref in KERNEL_SOURCES:
    candidates = _kernel_mount_candidates(ref)
    kaggle_input_paths[ref] = str(_first_existing(candidates) or candidates[0])

setup_env = {
    "TAAF_KAGGLE_INPUT_PATHS": json.dumps(kaggle_input_paths, sort_keys=True),
    "TAAF_KAGGLE_DATASET_SOURCES": json.dumps(DATASET_SOURCES),
    "TAAF_KAGGLE_KERNEL_SOURCES": json.dumps(KERNEL_SOURCES),
}
os.environ.update(setup_env)
_write_setup_env_updates(setup_env)
print(f"taaf.kaggle: input paths = {setup_env['TAAF_KAGGLE_INPUT_PATHS']}")

TAAF source bundle: /kaggle/input/datasets/lordhansolo/taaf-kaggle-source
taaf.kaggle: input paths = {"lordhansolo/taaf-kaggle-source": "/kaggle/input/datasets/lordhansolo/taaf-kaggle-source", "lordhansolo/vllm-main-e975732-arc3": "/kaggle/input/vllm-main-e975732-arc3"}


In [4]:
# Audit attached datasets
import subprocess

subprocess.run(["ls", "/kaggle/input"], check=False)

competitions
datasets
models
vllm-main-e975732-arc3


CompletedProcess(args=['ls', '/kaggle/input'], returncode=0)

In [5]:
def _source_path_entries(bundle_dir: Path) -> list[Path]:
    src_root = bundle_dir / "src"
    if not src_root.is_dir():
        return []
    entries: list[Path] = []
    for repo in sorted(src_root.iterdir(), reverse=True):
        if not repo.is_dir():
            continue
        for candidate in (repo / "src", repo):
            if candidate.is_dir():
                entries.append(candidate)
    return entries


def _command_env() -> dict[str, str]:
    env = os.environ.copy()
    env["PYTHON"] = sys.executable
    env["TAAF_KAGGLE_BUNDLE_DIR"] = str(BUNDLE_DIR)
    env["TAAF_KAGGLE_WORKING_DIR"] = str(WORKING_DIR)
    env["TAAF_KAGGLE_SETUP_ENV"] = str(SETUP_ENV_PATH)
    env.update(_load_setup_env())
    return env


def _run_shell_commands(filename: str, *, label: str, check: bool) -> None:
    path = BUNDLE_DIR / filename
    if not path.is_file():
        return
    commands = json.loads(path.read_text(encoding="utf-8"))
    env = _command_env()
    for command in commands:
        print(f"taaf.kaggle: {label} command: {command}", flush=True)
        result = subprocess.run(str(command), shell=True, check=check, cwd=WORKING_DIR, env=env)
        if not check and result.returncode != 0:
            print(f"taaf.kaggle: {label} command exited with {result.returncode}", flush=True)
        env.update(_load_setup_env())
        os.environ.update(env)


# Make bundled TAAF repos importable for this notebook and child Python processes.
source_entries = _source_path_entries(BUNDLE_DIR)
for entry in source_entries:
    sys.path.insert(0, str(entry))
if source_entries:
    import sysconfig

    pth_path = Path(sysconfig.get_paths()["purelib"]) / "taaf_kaggle_sources.pth"
    pth_path.write_text("".join(f"{entry}\n" for entry in source_entries), encoding="utf-8")
    print(f"taaf.kaggle: wrote {pth_path} ({len(source_entries)} source roots)", flush=True)

# Run deployment setup commands before the benchmark pickle is loaded.
_run_shell_commands("setup_commands.json", label="setup", check=True)

# Setup commands may export PYTHONPATH through TAAF_KAGGLE_SETUP_ENV.
pythonpath_entries = [entry for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep) if entry]
for entry in reversed(pythonpath_entries):
    if entry not in sys.path:
        sys.path.insert(0, entry)

taaf.kaggle: wrote /usr/local/lib/python3.12/dist-packages/taaf_kaggle_sources.pth (3 source roots)
taaf.kaggle: setup command: "$PYTHON" - <<'PYSETUP'
import json
import math
import os
import shutil
import subprocess
import sys
import time
import urllib.request
from datetime import datetime
from pathlib import Path


def is_test_run() -> bool:
    '''Return true only for an explicit noncompetition test run.'''

    return (
        os.environ.get('TAAF_RUN_AS_SUBMISSION', '').strip().lower() == '0'
        and os.environ.get('KAGGLE_IS_COMPETITION_RERUN', '').strip().lower() in {'', '0', 'false'}
    )


RUNTIME_OWNER = 'lordhansolo'
RUNTIME_SLUG = 'vllm-main-e975732-arc3'
MODEL_SOURCE_PARTS = ('lordhansolo', 'qwen3-8-flash-next-mixed-nvfp4-fp8', 'PyTorch', 'hf-mixed-mtp-nvfp4', '1')
SERVED_MODEL_NAME = 'primitive-ai/Qwen3.8-Flash-Next-mixed-NVFP4-FP8'
VLLM_HOST = '127.0.0.1'
VLLM_PORT = 1234
VLLM_BASE_URL = f'http://{VLLM_HOST}:{VLLM_PORT}/v1'
VLLM_SERVER_URL = f'http://{VLLM_HOST}:{

In [6]:
def _soft_end_time(max_runtime_s: float, *, run_as_submission: bool) -> datetime | None:
    if run_as_submission or max_runtime_s <= 0:
        return None
    budget = max(1.0, max_runtime_s)
    buffer = min(SOFT_DEADLINE_BUFFER_S, budget / 2)
    start = datetime.fromtimestamp(NOTEBOOK_START_EPOCH)
    return start + timedelta(seconds=budget - buffer)


def _competition_games():
    import arc_agi

    import taaf.game_api

    spec = taaf.game_api.ArcadeSpec(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=os.environ.get("ARC_BASE_URL", "http://gateway:8001/"),
        environments_dir="",
    )
    arcade = arc_agi.Arcade(
        operation_mode=arc_agi.OperationMode.COMPETITION,
        arc_base_url=spec.arc_base_url,
        environments_dir="",
    )
    game_ids = [env_info.game_id for env_info in arcade.available_environments]
    if not game_ids:
        raise RuntimeError("Competition Arcade exposed zero environments.")
    return [taaf.game_api.GameAPI(env_name=game_id, arcade_spec=spec) for game_id in game_ids]


@contextlib.contextmanager
def _tee_to_file(log_path: Path):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log_file = open(log_path, "w", buffering=1)
    original_stdout = sys.stdout
    original_stderr = sys.stderr
    sys.stdout = _Tee(original_stdout, log_file)
    sys.stderr = _Tee(original_stderr, log_file)
    try:
        yield
    finally:
        sys.stdout = original_stdout
        sys.stderr = original_stderr
        log_file.close()


class _Tee:
    def __init__(self, *streams: TextIO) -> None:
        self._streams = streams

    def write(self, data: str) -> int:
        n = 0
        for stream in self._streams:
            n = stream.write(data)
        return n

    def flush(self) -> None:
        for stream in self._streams:
            stream.flush()

    def isatty(self) -> bool:
        return any(getattr(stream, "isatty", lambda: False)() for stream in self._streams)

In [7]:
true_submission = _env_bool("KAGGLE_IS_COMPETITION_RERUN", False)
run_as_submission = _env_bool("TAAF_RUN_AS_SUBMISSION", False) or true_submission
os.environ["ONLY_RESET_LEVELS"] = "true"
os.environ["TAAF_RUN_AS_SUBMISSION"] = "1" if run_as_submission else "0"
os.environ["TAAF_MINIMAL_DIAGNOSTICS"] = "1" if run_as_submission else "0"

with open(BUNDLE_DIR / "deploy_target.pkl", "rb") as file:
    target = pickle.load(file)
target.actual_run_as_submission = run_as_submission
target.is_competition_rerun = true_submission
soft_end = _soft_end_time(float(getattr(target, "max_runtime_s", 0.0) or 0.0), run_as_submission=run_as_submission)

with open(BUNDLE_DIR / "benchmark_initial.pkl", "rb") as file:
    bm = pickle.load(file)
bm.job_dir = WORKING_DIR

In [8]:
# Inline customization hook.
# Make one-off changes to `bm`, `bm.games`, or `bm.solver` here before the run starts.
# Example:
# bm.label = f'{bm.label}-debug'


In [9]:
run_context = contextlib.nullcontext() if run_as_submission else _tee_to_file(WORKING_DIR / "stdout.log")
with run_context:
    preamble = (BUNDLE_DIR / "preamble.txt").read_text(encoding="utf-8")
    print(preamble)
    print(f"deploy.kaggle: working_dir             = {WORKING_DIR}")
    print(f"deploy.kaggle: run_as_submission       = {run_as_submission}")
    print(f"deploy.kaggle: competition_rerun       = {true_submission}")
    print(f"deploy.kaggle: soft_end_time           = {soft_end}")
    print("---")

    bundled_git_status = BUNDLE_DIR / "git_status.txt"
    if bundled_git_status.is_file():
        (WORKING_DIR / "git_status.txt").write_text(
            bundled_git_status.read_text(encoding="utf-8"),
            encoding="utf-8",
        )

    if true_submission:
        # Competition reruns use Kaggle's live gateway instead of the bundled offline games.
        os.environ.setdefault("ARC_API_KEY", "test-key-123")
        os.environ.setdefault("ARC_BASE_URL", "http://gateway:8001/")
        os.environ.setdefault("SCHEME", "http")
        os.environ.setdefault("HOST", "gateway")
        os.environ.setdefault("PORT", "8001")
        os.environ.setdefault("OPERATION_MODE", "competition")
        os.environ.setdefault("ENVIRONMENTS_DIR", "")
        os.environ.setdefault("RECORDINGS_DIR", str(WORKING_DIR / "server_recording"))

        deadline = time.monotonic() + 600.0
        last_error = ""
        while time.monotonic() < deadline:
            try:
                with urlopen("http://gateway:8001/api/games", timeout=10) as response:
                    if response.status < 500:
                        break
            except Exception as exc:
                last_error = repr(exc)
            time.sleep(5)
        else:
            raise RuntimeError(f"Kaggle gateway did not become ready: {last_error}")

        bm.games = _competition_games()
        bm.n_passes = 1
        bm.game_weights = None
    else:
        # This build ships no offline env files for the pickled ``__auto__`` specs;
        # on Kaggle, point them at the competition's bundled environment files.
        environment_files_dir = wheelhouse.parent / "environment_files"
        if environment_files_dir.is_dir():
            environments_dir = str(environment_files_dir)
            for game in bm.games:
                if game.arcade_spec.environments_dir == "__auto__":
                    game.arcade_spec = dataclasses.replace(
                        game.arcade_spec, environments_dir=environments_dir
                    )

    try:
        await bm.run(
            soft_end_time=soft_end,
            runtime_environment=target,
            minimal_diagnostics=run_as_submission,
        )
        if not true_submission and Path("/kaggle/input").exists():
            try:
                import pandas as pd

                submission = pd.DataFrame(
                    data=[["1_0", "1", True, 1]],
                    columns=["row_id", "game_id", "end_of_game", "score"],
                )
                submission.to_parquet(WORKING_DIR / "submission.parquet", index=False)
            except Exception as exc:
                print(f"taaf.kaggle: could not write offline dummy submission: {exc!r}", flush=True)
    finally:
        _run_shell_commands("teardown_commands.json", label="teardown", check=False)

benchmark.label : duck-harness-kaggle
benchmark.solver: HarnessSolver(label='duck-harness', runtime_environment=None, job_dir=None, soft_end_time=None, minimal_diagnostics=False, model='local', analyzer_timeout=900.0, max_actions_per_game=None, max_runtime_s_per_game=3918.0, concurrency=14, save_request_logs=False, start_local_server=False, local_server_config='', local_server_api_key_file='', local_server_repo_dir='/app/ARC3-Inference', local_server_port=None, local_server_tensor_parallel_size=None, local_server_count=1, cancel_drain_timeout_s=120.0)
benchmark.passes: 1
benchmark.games : 25
git status:
  ARC3-Inference                   ca1bd02     clean  solution-improve-qwen38-flash-next  Let the agent act without full certainty
  tufa-arc-agi-framework           ca1bd02     clean  solution-improve-qwen38-flash-next  Let the agent act without full certainty
deploy.kaggle: working_dir             = /kaggle/working
deploy.kaggle: run_as_submission       = False
deploy.kaggle: competit

analyzer request failed at action 49: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=20.0)
analyzer request failed at action 91: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=9.0)
analyzer request failed at action 56: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=6.0)
analyzer request failed at action 25: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=17.0)
analyzer request failed at action 119: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=113.0)
analyzer request failed at action 54: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=43.0)
analyzer request failed at action 81: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=25.0)
analyzer request failed at action 37: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=25.0)
analyzer request

[finished] lp85-305b61c3 state=gave_up level=3/8 score=16.67 actions=36 tokens=65345 per-level=5/17,11/38,17/31,3/16,0/41,0/60,0/26,0/159 note="tokens=65345"
[finished] ka59-38d34dbb state=gave_up level=2/7 score=10.71 actions=85 tokens=60759 per-level=26/28,59/109,0/51,0/51,0/33,0/132,0/326 note="tokens=60759"
[finished] sk48-d8078629 state=gave_up level=0/8 score=0.00 actions=0 tokens=0 per-level=0/61,0/177,0/101,0/103,0/230,0/181,0/125,0/92 note="tokens=0"
[finished] dc22-fdcac232 state=gave_up level=0/6 score=0.00 actions=0 tokens=0 per-level=0/59,0/102,0/67,0/98,0/324,0/578 note="tokens=0"
[finished] cd82-fb555c5d state=gave_up level=0/6 score=0.00 actions=0 tokens=0 per-level=0/55,0/8,0/41,0/21,0/23,0/23 note="tokens=0"
[finished] ft09-0d8bbf25 state=gave_up level=0/6 score=0.00 actions=0 tokens=0 per-level=0/43,0/12,0/23,0/28,0/65,0/37 note="tokens=0"
[finished] g50t-5849a774 state=gave_up level=0/7 score=0.00 actions=0 tokens=0 per-level=0/78,0/175,0/179,0/230,0/96,0/54,0/67 no

analyzer request failed at action 32: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=14.0)
analyzer request failed at action 69: HTTPConnectionPool(host='127.0.0.1', port=1234): Read timed out. (read timeout=83.0)


[finished] r11l-495a7899 state=gave_up level=2/6 score=14.29 actions=31 tokens=63167 per-level=9/22,22/33,0/51,0/26,0/52,0/49 note="tokens=63167"
[finished] vc33-5430563c state=gave_up level=3/7 score=21.43 actions=68 tokens=56900 per-level=12/7,14/18,32/44,10/61,0/131,0/34,0/152 note="tokens=56900"
[finished] bp35-0a0ad940 state=gave_up level=1/9 score=0.43 actions=48 tokens=59149 per-level=48/21,0/48,0/44,0/38,0/33,0/87,0/86,0/131,0/163 note="tokens=59149"
[finished] lf52-271a04aa state=gave_up level=2/10 score=5.45 actions=90 tokens=60946 per-level=19/32,71/81,0/60,0/71,0/205,0/148,0/244,0/109,0/164,0/225 note="tokens=60946"
[finished] sp80-589a99af state=gave_up level=1/6 score=4.76 actions=55 tokens=61280 per-level=16/39,39/58,0/25,0/148,0/96,0/152 note="tokens=61280"
[finished] cn04-2fe56bfb state=gave_up level=1/6 score=4.76 actions=24 tokens=61582 per-level=17/29,7/54,0/85,0/300,0/208,0/113 note="tokens=61582"
[finished] ar25-0c556536 state=gave_up level=4/8 score=27.78 actions